# 02. QLoRA Fine-tuning 학습

이 노트북에서는 Qwen2.5-7B 모델을 농업 도메인 Tool Calling에 특화시키기 위한 QLoRA Fine-tuning을 수행합니다.

## 목차
1. 환경 설정
2. 설정 로드
3. 모델 및 토크나이저 로드
4. LoRA 설정 및 적용
5. 데이터셋 준비
6. 학습 실행
7. 결과 저장

## 1. 환경 설정

In [ ]:
import os
import sys
sys.path.append('..')

import torch
import json
from dotenv import load_dotenv

# 환경 변수 로드
load_dotenv('../.env')

# GPU 확인
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# 필요한 라이브러리 임포트
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import load_dataset
import yaml

print("라이브러리 로드 완료!")

## 2. 설정 로드

실험 설정을 선택합니다:
- `exp-001`: Baseline (LR=2e-4, Rank=16, Epochs=3)
- `exp-002`: LR 최적화 (LR=1e-4, Rank=16, Epochs=5)
- `exp-003`: LoRA Rank 증가 (LR=1e-4, Rank=32, Epochs=5)

In [ ]:
# 실험 선택 (원하는 실험 번호로 변경)
EXPERIMENT = "exp-001"  # exp-001, exp-002, exp-003 중 선택

config_path = f'../configs/training_config_{EXPERIMENT.replace("-", "")}.yaml'

with open(config_path, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

print(f"실험: {config['experiment']['name']}")
print(f"설명: {config['experiment']['description']}")
print(f"\n주요 설정:")
print(f"  - Base Model: {config['model']['base_model']}")
print(f"  - Learning Rate: {config['training']['learning_rate']}")
print(f"  - Batch Size: {config['training']['per_device_train_batch_size']}")
print(f"  - LoRA Rank: {config['lora']['r']}")
print(f"  - Epochs: {config['training']['num_train_epochs']}")

## 3. 모델 및 토크나이저 로드

In [ ]:
model_config = config['model']

# BitsAndBytes 4-bit 양자화 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit=model_config['load_in_4bit'],
    bnb_4bit_quant_type=model_config['bnb_4bit_quant_type'],
    bnb_4bit_compute_dtype=getattr(torch, model_config['bnb_4bit_compute_dtype']),
    bnb_4bit_use_double_quant=model_config['bnb_4bit_use_double_quant'],
)

print("BitsAndBytes 설정 완료")
print(f"  - 4-bit quantization: {model_config['load_in_4bit']}")
print(f"  - Quantization type: {model_config['bnb_4bit_quant_type']}")
print(f"  - Compute dtype: {model_config['bnb_4bit_compute_dtype']}")

In [ ]:
print(f"모델 로딩 중: {model_config['base_model']}")
print("(처음 실행 시 모델 다운로드에 시간이 걸릴 수 있습니다)")

# 모델 로드
model = AutoModelForCausalLM.from_pretrained(
    model_config['base_model'],
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=model_config['trust_remote_code'],
)

print(f"\n모델 로드 완료!")
print(f"Model dtype: {model.dtype}")

In [ ]:
# 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained(
    model_config['base_model'],
    trust_remote_code=model_config['trust_remote_code'],
)

# Padding 설정
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"토크나이저 로드 완료!")
print(f"Vocab size: {tokenizer.vocab_size}")
print(f"Pad token: {tokenizer.pad_token}")

## 4. LoRA 설정 및 적용

In [ ]:
lora_config = config['lora']

# LoRA 설정
peft_config = LoraConfig(
    r=lora_config['r'],
    lora_alpha=lora_config['lora_alpha'],
    target_modules=lora_config['target_modules'],
    lora_dropout=lora_config['lora_dropout'],
    bias=lora_config['bias'],
    task_type=lora_config['task_type'],
)

print("LoRA 설정:")
print(f"  - Rank (r): {lora_config['r']}")
print(f"  - Alpha: {lora_config['lora_alpha']}")
print(f"  - Target modules: {lora_config['target_modules']}")
print(f"  - Dropout: {lora_config['lora_dropout']}")

In [ ]:
# 모델 준비 (k-bit training)
model = prepare_model_for_kbit_training(model)

# LoRA 적용
model = get_peft_model(model, peft_config)

# 학습 가능한 파라미터 확인
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in model.parameters())

print(f"\n학습 가능한 파라미터: {trainable_params:,} / {all_params:,}")
print(f"비율: {100 * trainable_params / all_params:.2f}%")

## 5. 데이터셋 준비

In [ ]:
data_config = config['data']

# JSON 데이터셋 로드
dataset = load_dataset('json', data_files={
    'train': f"../{data_config['train_file']}",
    'validation': f"../{data_config['val_file']}",
})

print(f"Train 샘플: {len(dataset['train'])}")
print(f"Validation 샘플: {len(dataset['validation'])}")
print(f"\n샘플 미리보기 (text 필드):")
print(dataset['train'][0]['text'][:500] + "...")

## 6. 학습 실행

In [ ]:
training_config = config['training']
misc_config = config['misc']
optimizer_config = config['optimizer']

# 출력 디렉토리
output_dir = f"../{misc_config['output_dir']}"
os.makedirs(output_dir, exist_ok=True)

# Training Arguments
training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=training_config['num_train_epochs'],
    per_device_train_batch_size=training_config['per_device_train_batch_size'],
    per_device_eval_batch_size=training_config.get('per_device_eval_batch_size', 4),
    gradient_accumulation_steps=training_config['gradient_accumulation_steps'],
    learning_rate=training_config['learning_rate'],
    weight_decay=training_config.get('weight_decay', 0.01),
    warmup_steps=training_config['warmup_steps'],
    logging_steps=training_config['logging_steps'],
    save_steps=training_config['save_steps'],
    eval_steps=training_config['eval_steps'],
    eval_strategy="steps",
    save_total_limit=training_config.get('save_total_limit', 3),
    load_best_model_at_end=training_config.get('load_best_model_at_end', True),
    fp16=misc_config.get('fp16', False),
    bf16=misc_config.get('bf16', True),
    max_grad_norm=misc_config.get('max_grad_norm', 0.3),
    optim=optimizer_config.get('optim', 'paged_adamw_8bit'),
    lr_scheduler_type=optimizer_config.get('lr_scheduler_type', 'cosine'),
    seed=misc_config.get('seed', 42),
    report_to=misc_config.get('report_to', 'none'),
    logging_dir=os.path.join(output_dir, 'logs'),
)

print("Training Arguments 설정 완료")

In [ ]:
# SFTTrainer 생성
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    tokenizer=tokenizer,
    max_seq_length=data_config.get('max_seq_length', 2048),
    dataset_text_field=data_config.get('dataset_text_field', 'text'),
)

print("SFTTrainer 생성 완료")
print(f"Max sequence length: {data_config.get('max_seq_length', 2048)}")

In [ ]:
# 학습 시작
print("="*60)
print(f"학습 시작: {config['experiment']['name']}")
print("="*60)

trainer.train()

print("\n" + "="*60)
print("학습 완료!")
print("="*60)

## 7. 결과 저장

In [ ]:
# 모델 저장
final_checkpoint_dir = os.path.join(output_dir, "final_checkpoint")

print(f"모델 저장 중: {final_checkpoint_dir}")
trainer.save_model(final_checkpoint_dir)
tokenizer.save_pretrained(final_checkpoint_dir)

print("모델 저장 완료!")

In [ ]:
# 학습 로그 저장
log_history = trainer.state.log_history
log_path = os.path.join(output_dir, "training_log.json")

with open(log_path, 'w') as f:
    json.dump(log_history, f, indent=2)

print(f"학습 로그 저장: {log_path}")

In [ ]:
# Loss Curve 시각화
import matplotlib.pyplot as plt

train_loss = [(x['step'], x['loss']) for x in log_history if 'loss' in x and 'eval_loss' not in x]
eval_loss = [(x['step'], x['eval_loss']) for x in log_history if 'eval_loss' in x]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training Loss
if train_loss:
    steps, losses = zip(*train_loss)
    axes[0].plot(steps, losses, label='Train Loss', color='blue')
    axes[0].set_xlabel('Steps')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

# Validation Loss
if eval_loss:
    steps, losses = zip(*eval_loss)
    axes[1].plot(steps, losses, label='Eval Loss', color='orange')
    axes[1].set_xlabel('Steps')
    axes[1].set_ylabel('Loss')
    axes[1].set_title('Validation Loss')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

plt.suptitle(f"{config['experiment']['name']} Loss Curve", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'loss_curve.png'), dpi=300)
plt.show()

print(f"Loss curve 저장: {output_dir}/loss_curve.png")

In [ ]:
# 최종 결과 요약
final_train_loss = train_loss[-1][1] if train_loss else None
final_eval_loss = eval_loss[-1][1] if eval_loss else None

print("\n" + "="*60)
print(f"실험 완료: {config['experiment']['name']}")
print("="*60)
print(f"\n최종 결과:")
print(f"  - Final Train Loss: {final_train_loss:.4f}" if final_train_loss else "  - Train Loss: N/A")
print(f"  - Final Eval Loss: {final_eval_loss:.4f}" if final_eval_loss else "  - Eval Loss: N/A")
print(f"\n저장된 파일:")
print(f"  - 체크포인트: {final_checkpoint_dir}")
print(f"  - 학습 로그: {log_path}")
print(f"  - Loss Curve: {output_dir}/loss_curve.png")

## 다음 단계

학습이 완료되었습니다. 다음 노트북에서 Tool Calling 정확도를 평가합니다:
- `03_evaluation.ipynb`